# 6b: Per-Sector GCN vs GAT Breakdown

Notebook 6 Section 3 reports only aggregate sector statistics (intra/inter ratio 2.12 vs 1.86; modularity 0.085 vs 0.066; modularity↔Sharpe r≈0). These averages collapse across 10 GICS sectors, so any asymmetric effect — e.g. Financials tightening while Utilities loosens — cancels in the mean.

This notebook decomposes those aggregates per sector to answer:

1. Under GAT, is Financials still the densest intra-sector cluster, or does attention reshuffle the ranking?
2. Which sectors does GAT tighten vs loosen relative to Pearson?
3. Where do GAT − GCN position differences concentrate, and do those sectors drive the COVID Sharpe advantage?
4. Does attention asymmetry cluster in certain sectors?

## 1. Setup & Artefact Loading

Replicates notebook 6's loading pattern exactly: same experiments, same seeds, same shared helper assumptions.

In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        get_ipython().system('git clone https://github.com/adam-909/4yp.git /content/repo')
    else:
        get_ipython().system('cd /content/repo && git pull')
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
elif os.path.exists("/mnt/g/My Drive"):
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "/mnt/g/My Drive/FINAL_RESULTS"
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"

sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")
print(f"Results base: {RESULTS_BASE}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from gml.experiment_utils import load_experiment_results
from gml.graph_visualization import SECTOR_COLORS
from settings.default import ALL_TICKERS, BBG_SECTORS

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "figure.facecolor": "white",
})
print("Imports ready")

In [ ]:
# Same experiments / seeds as notebook 6
GCN_EXPERIMENT = "3_GCN_rolling_pearson"
GCN_CONFIG = "lb20_th0.4_equity"
GCN_SEED = 42

GAT_EXPERIMENT = "4c_GATv2_rolling_pearson_th0.4_scFalse_resTrue"
GAT_CONFIG = "lb20_th0.4_equity"
GAT_SEED = 40

print(f"Loading GCN: {GCN_EXPERIMENT}/{GCN_CONFIG}/seed_{GCN_SEED}")
gcn_results = load_experiment_results(GCN_EXPERIMENT, GCN_SEED, config_name=GCN_CONFIG, base_dir=RESULTS_BASE)
print(f"Loading GAT: {GAT_EXPERIMENT}/{GAT_CONFIG}/seed_{GAT_SEED}")
gat_results = load_experiment_results(GAT_EXPERIMENT, GAT_SEED, config_name=GAT_CONFIG, base_dir=RESULTS_BASE)

gcn_adj = gcn_results["adjacency"]          # (W, 88, 88)
gcn_daily = gcn_results["daily_returns"]
gcn_cr = gcn_results["captured_returns"]
gcn_dates = gcn_results["test_dates"]

gat_attn = gat_results["attention_weights"]   # (W, 4, 88, 88)
gat_adj = gat_results["adjacency"]
gat_daily = gat_results["daily_returns"]
gat_cr = gat_results["captured_returns"]
gat_dates = gat_results["test_dates"]

gat_attn_avg = gat_attn.mean(axis=1)          # (W, 88, 88)

tickers = sorted(ALL_TICKERS)
sectors = [BBG_SECTORS.get(t, "Unknown") for t in tickers]
unique_sectors = sorted(set(sectors))
sector_arr = np.array(sectors)

print(f"\nGCN adj: {gcn_adj.shape}   GAT attn: {gat_attn.shape}   GAT avg: {gat_attn_avg.shape}")
print(f"Tickers: {len(tickers)}   Sectors ({len(unique_sectors)}): {unique_sectors}")

In [ ]:
# VIX + regime (reused for stress splits in sections 3 and 4)
import yfinance as yf
vix = yf.download("^VIX", start="2017-01-01", end="2023-09-01")["Close"].squeeze()
vix.index = pd.to_datetime(vix.index)

gcn_dates_dt = pd.to_datetime(gcn_dates)
gat_dates_dt = pd.to_datetime(gat_dates)
vix_gcn = vix.reindex(gcn_dates_dt, method="ffill")
vix_gat = vix.reindex(gat_dates_dt, method="ffill")
vix_q25, vix_q75 = vix_gcn.quantile(0.25), vix_gcn.quantile(0.75)

def regime_series(vix_aligned):
    r = pd.Series("Mid", index=vix_aligned.index)
    r[vix_aligned > vix_q75] = "High"
    r[vix_aligned < vix_q25] = "Low"
    return r

regime_gcn = regime_series(vix_gcn)
regime_gat = regime_series(vix_gat)
print(f"VIX Q25={vix_q25:.1f}, Q75={vix_q75:.1f}")
print(f"GAT regime counts: {regime_gat.value_counts().to_dict()}")

In [ ]:
# Sanity-check: reproduce notebook 6's aggregate intra/inter ratio and modularity
# Sector masks as boolean 88x88 arrays
same_sector = sector_arr[:, None] == sector_arr[None, :]
np.fill_diagonal(same_sector, False)
intra_mask = same_sector
inter_mask = ~same_sector
np.fill_diagonal(inter_mask, False)

def intra_inter_ratio(A_series):
    ratios = []
    for A in A_series:
        a = A.copy(); np.fill_diagonal(a, 0)
        intra_d = a[intra_mask].mean()
        inter_d = a[inter_mask].mean()
        ratios.append(intra_d / inter_d if inter_d > 0 else np.nan)
    return np.array(ratios)

def newman_modularity(A_series, membership):
    """Weighted Newman modularity with predefined community labels."""
    Qs = []
    for A in A_series:
        a = A.copy(); np.fill_diagonal(a, 0)
        m2 = a.sum()
        if m2 == 0:
            Qs.append(np.nan); continue
        k = a.sum(axis=1)
        same = membership[:, None] == membership[None, :]
        Q = ((a - np.outer(k, k) / m2) * same).sum() / m2
        Qs.append(Q)
    return np.array(Qs)

gcn_ratio = intra_inter_ratio(gcn_adj)
gat_ratio = intra_inter_ratio(gat_attn_avg)
gcn_Q = newman_modularity(gcn_adj, sector_arr)
gat_Q = newman_modularity(gat_attn_avg, sector_arr)

print("Aggregate sanity check (should roughly match notebook 6):")
print(f"  Intra/inter ratio  GCN: {np.nanmean(gcn_ratio):.3f} ± {np.nanstd(gcn_ratio):.3f}  (nb6: 2.12)")
print(f"  Intra/inter ratio  GAT: {np.nanmean(gat_ratio):.3f} ± {np.nanstd(gat_ratio):.3f}  (nb6: 1.86)")
print(f"  Newman modularity  GCN: {np.nanmean(gcn_Q):.4f} ± {np.nanstd(gcn_Q):.4f}  (nb6: 0.085)")
print(f"  Newman modularity  GAT: {np.nanmean(gat_Q):.4f} ± {np.nanstd(gat_Q):.4f}  (nb6: 0.066)")

## 2. Per-sector intra-density table

For each GICS sector compute mean intra-sector edge weight under GCN (Pearson) and GAT (attention), along with rank and GAT/GCN ratio. **Answers Q1 & Q2.**

In [ ]:
def per_sector_intra_density(A_series):
    """Mean intra-sector edge weight per sector, averaged over all windows."""
    out = {s: [] for s in unique_sectors}
    idx_by_sector = {s: np.where(sector_arr == s)[0] for s in unique_sectors}
    for A in A_series:
        a = A.copy(); np.fill_diagonal(a, 0)
        for s, idxs in idx_by_sector.items():
            if len(idxs) < 2:
                out[s].append(np.nan); continue
            block = a[np.ix_(idxs, idxs)]
            iu = np.triu_indices(len(idxs), k=1)
            out[s].append(block[iu].mean())
    return {s: np.nanmean(v) for s, v in out.items()}, {s: np.array(v) for s, v in out.items()}

gcn_intra_mean, gcn_intra_series = per_sector_intra_density(gcn_adj)
gat_intra_mean, gat_intra_series = per_sector_intra_density(gat_attn_avg)

# Sector size + intra-edge counts under Pearson mask (comparable to 5b's 84-109 for Financials)
rows = []
for s in unique_sectors:
    idxs = np.where(sector_arr == s)[0]
    n = len(idxs)
    pair_count = n * (n - 1) // 2
    # Mean intra-sector Pearson edge count per window (A>0 within sector block)
    counts = []
    for A in gcn_adj:
        block = A[np.ix_(idxs, idxs)].copy(); np.fill_diagonal(block, 0)
        counts.append((block > 0).sum() / 2)
    rows.append({
        "Sector": s, "N": n, "Pairs": pair_count,
        "GCN intra-weight": gcn_intra_mean[s],
        "GAT intra-weight": gat_intra_mean[s],
        "GAT/GCN ratio": gat_intra_mean[s] / gcn_intra_mean[s] if gcn_intra_mean[s] else np.nan,
        "GCN edges (mean)": np.mean(counts),
        "GCN edges (max)": np.max(counts),
    })

intra_df = pd.DataFrame(rows).sort_values("GCN intra-weight", ascending=False).reset_index(drop=True)
intra_df["GCN rank"] = intra_df["GCN intra-weight"].rank(ascending=False).astype(int)
intra_df["GAT rank"] = intra_df["GAT intra-weight"].rank(ascending=False).astype(int)
intra_df["Rank Δ"] = intra_df["GAT rank"] - intra_df["GCN rank"]
print("Per-sector intra-density (sorted by GCN weight):\n")
print(intra_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Horizontal bar chart: GAT/GCN ratio per sector (>1 tightens, <1 loosens)
df_plot = intra_df.sort_values("GAT/GCN ratio")
fig, ax = plt.subplots(figsize=(9, 5))
colors = [SECTOR_COLORS.get(s, "gray") for s in df_plot["Sector"]]
bars = ax.barh(df_plot["Sector"], df_plot["GAT/GCN ratio"], color=colors, edgecolor="black")
ax.axvline(1.0, color="black", ls="--", lw=1, label="GAT = GCN")
ax.set_xlabel("GAT intra-density / GCN intra-density")
ax.set_title("Where does GAT tighten vs loosen sector cohesion?")
for bar, ratio in zip(bars, df_plot["GAT/GCN ratio"]):
    ax.text(ratio + 0.01, bar.get_y() + bar.get_height()/2, f"{ratio:.2f}", va="center", fontsize=9)
ax.legend()
plt.tight_layout()
plt.show()

# Ranking changes
changed = intra_df[intra_df["Rank Δ"] != 0]
if len(changed):
    print("\nSectors whose intra-density rank changes between GCN and GAT:")
    print(changed[["Sector", "GCN rank", "GAT rank", "Rank Δ"]].to_string(index=False))
else:
    print("\nNo rank changes — GAT preserves Pearson's sector ordering.")

print(f"\nFinancials under GCN: rank {int(intra_df[intra_df.Sector=='Financials']['GCN rank'].iloc[0])}")
print(f"Financials under GAT: rank {int(intra_df[intra_df.Sector=='Financials']['GAT rank'].iloc[0])}")

## 3. Per-sector Newman modularity decomposition

Newman modularity decomposes naturally per community:
$$Q = \sum_s Q_s, \qquad Q_s = \frac{1}{2m} \sum_{i,j \in s} \left(A_{ij} - \frac{k_i k_j}{2m}\right)$$

Per-window $Q_s$ is averaged over time. The sum over sectors must equal the aggregate modularity from Section 1 (sanity-check).

In [ ]:
def per_sector_modularity(A_series, membership):
    Qs_by_sector = {s: [] for s in unique_sectors}
    idx_by_sector = {s: np.where(membership == s)[0] for s in unique_sectors}
    for A in A_series:
        a = A.copy(); np.fill_diagonal(a, 0)
        m2 = a.sum()
        if m2 == 0:
            for s in unique_sectors:
                Qs_by_sector[s].append(np.nan)
            continue
        k = a.sum(axis=1)
        for s, idxs in idx_by_sector.items():
            block_A = a[np.ix_(idxs, idxs)]
            block_E = np.outer(k[idxs], k[idxs]) / m2
            Qs_by_sector[s].append((block_A - block_E).sum() / m2)
    return {s: np.array(v) for s, v in Qs_by_sector.items()}

gcn_Qs = per_sector_modularity(gcn_adj, sector_arr)
gat_Qs = per_sector_modularity(gat_attn_avg, sector_arr)

mod_rows = []
for s in unique_sectors:
    mod_rows.append({
        "Sector": s,
        "GCN Q_s": np.nanmean(gcn_Qs[s]),
        "GAT Q_s": np.nanmean(gat_Qs[s]),
        "Δ (GAT−GCN)": np.nanmean(gat_Qs[s]) - np.nanmean(gcn_Qs[s]),
    })
mod_df = pd.DataFrame(mod_rows).sort_values("GCN Q_s", ascending=False).reset_index(drop=True)

print("Per-sector Newman modularity contribution:\n")
print(mod_df.to_string(index=False, float_format="%.4f"))
print(f"\nSum-check (should match aggregates):")
print(f"  GCN Σ Q_s = {mod_df['GCN Q_s'].sum():.4f}   aggregate: {np.nanmean(gcn_Q):.4f}")
print(f"  GAT Σ Q_s = {mod_df['GAT Q_s'].sum():.4f}   aggregate: {np.nanmean(gat_Q):.4f}")

In [ ]:
# VIX-regime split: does GAT flatten sectors uniformly, or selectively?
def mean_by_regime(series_dict, regime):
    out = {}
    for s, arr in series_dict.items():
        sub = pd.Series(arr, index=regime.index)
        out[s] = {r: sub[regime == r].mean() for r in ["Low", "Mid", "High"]}
    return pd.DataFrame(out).T

gcn_regime_Q = mean_by_regime(gcn_Qs, regime_gcn)
gat_regime_Q = mean_by_regime(gat_Qs, regime_gat)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, df, title in [(axes[0], gcn_regime_Q, "GCN Q_s by VIX regime"),
                       (axes[1], gat_regime_Q, "GAT Q_s by VIX regime")]:
    df_sorted = df.loc[mod_df["Sector"]]
    df_sorted[["Low", "Mid", "High"]].plot.barh(ax=ax, width=0.75,
                                                   color=["#2ca02c", "#ff7f0e", "#d62728"])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel("Mean Q_s")
plt.tight_layout()
plt.show()

print("\nDelta Q_s (GAT − GCN) by VIX regime — negative means GAT flattens this sector more in that regime:")
delta_regime = (gat_regime_Q - gcn_regime_Q).loc[mod_df["Sector"]]
print(delta_regime.to_string(float_format="%.4f"))

## 4. Per-sector position attribution

Split the 32–52% position shrinkage noted in notebook 6 into per-sector components. Per-sector Sharpe of (GAT − GCN) captured returns pinpoints which sectors drove the COVID outperformance.

In [ ]:
# Reuse notebook 6's pivot approach
def pivot_positions(cr_df):
    df = cr_df.copy(); df["time"] = pd.to_datetime(df["time"])
    return df.pivot_table(index="time", columns="identifier", values="position", aggfunc="first")

def pivot_capret(cr_df):
    df = cr_df.copy(); df["time"] = pd.to_datetime(df["time"])
    return df.pivot_table(index="time", columns="identifier", values="captured_returns", aggfunc="first")

gcn_pos = pivot_positions(gcn_cr)
gat_pos = pivot_positions(gat_cr)
gcn_capret = pivot_capret(gcn_cr)
gat_capret = pivot_capret(gat_cr)

common_dates = gcn_pos.index.intersection(gat_pos.index)
common_tickers = [t for t in tickers if t in gcn_pos.columns and t in gat_pos.columns]
gcn_pos = gcn_pos.loc[common_dates, common_tickers]
gat_pos = gat_pos.loc[common_dates, common_tickers]
gcn_capret = gcn_capret.reindex(index=common_dates, columns=common_tickers)
gat_capret = gat_capret.reindex(index=common_dates, columns=common_tickers)

ticker_to_sector = {t: BBG_SECTORS.get(t, "Unknown") for t in common_tickers}
print(f"Aligned: {gcn_pos.shape[0]} dates × {gcn_pos.shape[1]} tickers")

In [ ]:
# Per-sector position magnitudes, signed diffs, and Sharpe of (GAT - GCN) return differential
pos_rows = []
for s in unique_sectors:
    cols = [t for t in common_tickers if ticker_to_sector[t] == s]
    if not cols:
        continue
    gcn_mag = gcn_pos[cols].abs().mean().mean()
    gat_mag = gat_pos[cols].abs().mean().mean()
    signed  = (gat_pos[cols] - gcn_pos[cols]).mean().mean()
    # Daily sector-aggregate captured return
    gcn_daily_s = gcn_capret[cols].sum(axis=1)
    gat_daily_s = gat_capret[cols].sum(axis=1)
    diff_daily  = (gat_daily_s - gcn_daily_s).dropna()
    sharpe_diff = diff_daily.mean() / diff_daily.std() * np.sqrt(252) if diff_daily.std() > 0 else np.nan
    pos_rows.append({
        "Sector": s, "N": len(cols),
        "GCN |pos|": gcn_mag, "GAT |pos|": gat_mag,
        "GAT/GCN |pos|": gat_mag / gcn_mag if gcn_mag else np.nan,
        "Mean signed Δ": signed,
        "Sharpe(GAT−GCN)": sharpe_diff,
    })
pos_df = pd.DataFrame(pos_rows).sort_values("Sharpe(GAT−GCN)", ascending=False).reset_index(drop=True)
print("Per-sector position attribution (full test window):\n")
print(pos_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Stress-period split: does sector attribution change during COVID vs 2022 rate hikes?
stress_periods = {
    "COVID (Q1-Q2 2020)": ("2020-01-01", "2020-06-30"),
    "Rate Hikes (Q1-Q3 2022)": ("2022-01-01", "2022-09-30"),
}
stress_tables = {}
for name, (start, end) in stress_periods.items():
    mask = (common_dates >= pd.Timestamp(start)) & (common_dates <= pd.Timestamp(end))
    dates_p = common_dates[mask]
    rows = []
    for s in unique_sectors:
        cols = [t for t in common_tickers if ticker_to_sector[t] == s]
        if not cols: continue
        diff_daily = (gat_capret.loc[dates_p, cols].sum(axis=1) - gcn_capret.loc[dates_p, cols].sum(axis=1)).dropna()
        sharpe_diff = diff_daily.mean() / diff_daily.std() * np.sqrt(252) if diff_daily.std() > 0 else np.nan
        rows.append({"Sector": s, "Days": len(dates_p), "Sharpe(GAT−GCN)": sharpe_diff,
                     "Mean Δreturn (bps)": diff_daily.mean() * 1e4})
    stress_tables[name] = pd.DataFrame(rows).sort_values("Sharpe(GAT−GCN)", ascending=False).reset_index(drop=True)

for name, tbl in stress_tables.items():
    print(f"\n═══ {name} ═══")
    print(tbl.to_string(index=False, float_format="%.4f"))

In [ ]:
# Visual: side-by-side per-sector Sharpe(GAT−GCN) full test vs COVID vs 2022
combined = pos_df[["Sector", "Sharpe(GAT−GCN)"]].rename(columns={"Sharpe(GAT−GCN)": "Full"}).set_index("Sector")
combined["COVID"] = stress_tables["COVID (Q1-Q2 2020)"].set_index("Sector")["Sharpe(GAT−GCN)"]
combined["Rates 2022"] = stress_tables["Rate Hikes (Q1-Q3 2022)"].set_index("Sector")["Sharpe(GAT−GCN)"]
combined = combined.sort_values("COVID", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5.5))
combined.plot.barh(ax=ax, width=0.8, color=["#1f77b4", "#d62728", "#ff7f0e"])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Sharpe(GAT − GCN)")
ax.set_title("Per-sector contribution to GAT–GCN Sharpe differential")
ax.legend(title="Period", loc="lower right")
plt.tight_layout()
plt.show()

## 5. Per-sector attention asymmetry

Surfaces the numbers that notebook 6 plots but never prints: mean intra-sector |A[i,j] − A[j,i]|, plus inter-sector out-flow / in-flow / net per sector.

In [ ]:
# Pre-compute mean attention matrix (average over windows)
A_bar = gat_attn_avg.mean(axis=0)   # (88, 88)
np.fill_diagonal(A_bar, 0)
asym_mat = np.abs(A_bar - A_bar.T)

asym_rows = []
for s in unique_sectors:
    idxs = np.where(sector_arr == s)[0]
    other = np.where(sector_arr != s)[0]
    # Intra: upper triangle of within-sector block
    intra_block = asym_mat[np.ix_(idxs, idxs)]
    iu = np.triu_indices(len(idxs), k=1)
    intra_asym = intra_block[iu].mean() if len(idxs) >= 2 else np.nan
    # Inter flows (directional)
    out_flow = A_bar[np.ix_(idxs, other)].sum()   # attention FROM sector s TO others
    in_flow  = A_bar[np.ix_(other, idxs)].sum()   # attention TO sector s FROM others
    asym_rows.append({
        "Sector": s, "N": len(idxs),
        "Intra |ΔA|": intra_asym,
        "Out-flow": out_flow, "In-flow": in_flow,
        "Net (out−in)": out_flow - in_flow,
    })
asym_df = pd.DataFrame(asym_rows).sort_values("Intra |ΔA|", ascending=False).reset_index(drop=True)
print("Per-sector attention asymmetry (time-averaged):\n")
print(asym_df.to_string(index=False, float_format="%.5f"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
df_a = asym_df.sort_values("Intra |ΔA|")
axes[0].barh(df_a["Sector"], df_a["Intra |ΔA|"],
             color=[SECTOR_COLORS.get(s, "gray") for s in df_a["Sector"]], edgecolor="black")
axes[0].set_xlabel("Mean intra-sector |A[i,j] − A[j,i]|")
axes[0].set_title("Intra-sector asymmetry")

df_b = asym_df.sort_values("Net (out−in)")
axes[1].barh(df_b["Sector"], df_b["Net (out−in)"],
             color=["#d62728" if v < 0 else "#2ca02c" for v in df_b["Net (out−in)"]], edgecolor="black")
axes[1].axvline(0, color="black", lw=0.8)
axes[1].set_xlabel("Net directional attention (out − in)")
axes[1].set_title("Which sectors broadcast vs absorb attention?")
plt.tight_layout()
plt.show()

## 6. Combined 10-sector summary

Single heatmap: 10 sectors × 5 metrics, z-scored per column, red/blue diverging. Each row shows where that sector sits on the GCN↔GAT axis across every metric.

In [ ]:
summary = pd.DataFrame({"Sector": unique_sectors}).set_index("Sector")
summary["Intra-density ratio"] = intra_df.set_index("Sector")["GAT/GCN ratio"]
summary["Δ Q_s"]              = mod_df.set_index("Sector")["Δ (GAT−GCN)"]
summary["Δ |pos|"]             = pos_df.set_index("Sector")["GAT |pos|"] - pos_df.set_index("Sector")["GCN |pos|"]
summary["Sharpe(GAT−GCN)"]     = pos_df.set_index("Sector")["Sharpe(GAT−GCN)"]
summary["Net asymmetry"]        = asym_df.set_index("Sector")["Net (out−in)"]
summary = summary.sort_values("Sharpe(GAT−GCN)", ascending=False)

z = (summary - summary.mean()) / summary.std()

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(z, annot=summary.round(4), fmt="", cmap="RdBu_r", center=0,
            cbar_kws={"label": "z-score (column-wise)"}, ax=ax, linewidths=0.5)
ax.set_title("Per-sector GCN vs GAT summary\n(cell values = raw; colour = column z-score)")
plt.tight_layout()
plt.show()

print("\nRaw summary table:\n")
print(summary.to_string(float_format="%.4f"))

## Findings

Populate from the printed tables above:

- **Q1 — Financials under GAT:** see Section 2 ranking table. If Financials rank changes, aggregate-means narrative is misleading.
- **Q2 — Tighten vs loosen:** sectors with ratio >1 in Section 2 are tightened by attention; <1 are loosened.
- **Q3 — COVID attribution:** top Sharpe(GAT−GCN) sectors in the COVID stress table drove the +1.28 aggregate advantage.
- **Q4 — Asymmetry concentration:** Section 5 surfaces which sectors have the highest intra-asymmetry and whether any sector systematically broadcasts/absorbs attention.

The Section 6 heatmap is the one-glance summary: any row that is strongly coloured across multiple columns is a sector where GCN↔GAT differences are large and coherent, not noise.